In [62]:

from dotenv import load_dotenv
from openai import OpenAI
import os
from pypdf import PdfReader
from pydantic import BaseModel

In [63]:

load_dotenv(override=True)

myOpenAI = OpenAI()

In [64]:
# pobieramy pdf i zapętlamy go na page i metoda extract przekauzjemy do zmienenj

pdfSummary = ''
pdfFile = PdfReader('cesarstwo_rzymskie.pdf')

for page in pdfFile.pages:
    text = page.extract_text()
    if (text):
        pdfSummary += text


print(pdfSummary)

Cesarstwo Rzymskie
 Zarys historyczny
1. Powstanie Cesarstwa
Cesarstwo Rzymskie powstało w 27 roku p.n.e., gdy senat rzymski nadał
Oktawianowi tytuł Augusta, kończąc tym samym epokę Republiki Rzymskiej.
Wcześniejszy okres naznaczony był długotrwałymi wojnami domowymi, które
rozdarły rzymskie społeczeństwo: konflikt Mariusza i Sulli, dyktatura Juliusza
Cezara zakończona jego zamachem w 44 roku p.n.e. w idy marcowe, a
następnie wojna między Markiem Antoniuszem a Oktawianem, rozstrzygnięta
w bitwie pod Akcjum w 31 roku p.n.e. Zwycięstwo Oktawiana otworzyło drogę
do przekształcenia ustroju państwa.
August formalnie zachował instytucje republikańskie - senat, konsulów,
zgromadzenia ludowe - lecz skupił w swoich rękach realną władzę wojskową,
finansową i religijną. Ten ustrój historycy nazywają pryncypatem. August
twierdził, że jest jedynie pierwszym obywatelem (princeps), choć w
rzeczywistości sprawował władzę monarszą. Jego długie panowanie (do 14
roku n.e.) zapewniło państwu stabilność i 

In [65]:

# piszemy system_prompt 

system_prompt = f'jestes modelem ai, który ma odpowiedać na pytania uzytkownika o Cesarstwo rzymskie'
system_prompt += f' masz korzystać z pliku pdf, który masz podany  w {pdfSummary}, odpowiadaj zgodnie z prawdą, jak prawdziwy rzymianin!!'
system_prompt += f' mów tylko prawde, i dawaj rzymskie wstawki '

In [66]:
# piszemy klase ooartą o base model z biblioteki pydantic do której zwrócimy odpowiedz modelu evaluator ta odpowiedź zostanie zwrócona zgodnie z 2 polami  
# 1 pole zwróci nam true lub false 2 da feedback w tej formie model evalautoar zwróci nam informację a ta kalsa evaluitaon nam ją po prostu odpowiednio poda

class Evaluation(BaseModel):
    is_accept: bool
    feedback: str

In [67]:
# piszemy prompt dla modelu evalautor który sprawdza 1 odpowiedz modelu chat i ma nam zwrócić w json true false oraz dac feedback dlaczego uwaza ze ta odpowiedz jest donra albo zła

evaluator_system_prompt = f"""
Jesteś ewaluatorem odpowiedzi AI.

Twoim zadaniem jest ocenić, czy odpowiedź jest zgodna z dostarczonymi materiałami dotyczącymi cesarstwa rzymskiego.



Jeśli odpowiedź zawiera błędy, nieścisłości lub wychodzi poza kontekst — oznacz ją jako nieakceptowalną.

---

## Materiały:


### PDF:
{pdfSummary}

---

Zwróć WYŁĄCZNIE JSON w formacie:
 is_acceptable (bool), feedback (string)
 """

In [68]:
# piszemy evalautor_user_prompt jako funkcje, przyjmuje ona reply - odpowiedź chatu 1 modelu message, wiadomosc usera oraz obiekt history
# piszemy go jako funkcje zeby móc uzyc wiecej niz 1 raz, za kazdym gdy wywołamy chat a potem evalautor do tej funkcji trafi nowe reply, message i uzupelnione o dane history
# ta funkcja jest wywoływana w evalautor // z niej trafiają tutaj nowe rep;y message i uzupełnione history

def evaluator_user_prompt(reply,message,history):
    evalautor_prompt = f'tutaj masz odpowiedź modelu {reply}'
    evalautor_prompt += f'tutaj wiadomość od usera {message}'
    evalautor_prompt += f'tutaj historie rozmowy {history}'
    evalautor_prompt += f'oceń odpowiedź modelu'
    return evalautor_prompt

In [69]:

# funkcja evalautor funkcja uruchamiuana w chat, przekazujemreply message history do evalautor suser prompt

def evaluator(reply,message,history) -> Evaluation:
    messages = [{'role':'system',"content":evaluator_system_prompt}] + [{'role':'user','content':evaluator_user_prompt(reply,message,history)}]

    response = myOpenAI.beta.chat.completions.parse(
        model='gpt-4.1-mini',
        messages=messages,
        response_format=Evaluation
    )
    return response.choices[0].message.parsed



In [70]:

# funkcja rerun uruchamia się tylko jesli evalautor zwróci false, i BaseModel to false zapisze w is_acceptalbe funkcja uruchamiaana w chat

def rerun(reply,message,history,feedback):

    new_prompt = system_prompt + f'poprzednia wiadomość została odrzucona'
    new_prompt += f'tu masz odpowiedz modelu {reply}'
    new_prompt += f' feedback {feedback}'

    messages = [{'role':'system','content':new_prompt}] + history + [{'role':'user',"content":message}]

    response = myOpenAI.chat.completions.create(
        model='gpt-4.1-mini',
        messages=messages
    )

    return response.choices[0].message.content


In [71]:
 # funkcja chat uruchamia evalautor i rerun

def chat(message,history):

    messages = [{'role':'system',"content":system_prompt}] + history +  [{'role':'user',"content":message}]
    # tworzymy model



    response = myOpenAI.chat.completions.create(

        model='gpt-4.1-mini',
        messages =messages
    )

    reply = response.choices[0].message.content

    evaluation = evaluator(reply,message,history)

    if (evaluation.is_accept):
        print(evaluation.is_accept)
        print(evaluation.feedback)
        print(reply)
        print(evaluator(reply,message,history))

    else: 
        print(evaluation.is_accept)
        print(evaluation.feedback)
        reply = rerun(reply,message,history,evaluation.feedback)
        print(evaluator(reply,message,history))
    return reply     # ten return trafia do 

In [72]:

history = []

msg1  = 'powiedz coś głupiego o cesarstwie'

reply1 = chat(msg1,history)  

# appemd do history 

history.append({'role':'user','content':msg1})
history.append({'role':'assistant','content':reply1})


# print 

print(reply1)


False
Odpowiedź jest emocjonalna i nie zawiera rzetelnych informacji historycznych na temat Cesarstwa Rzymskiego. Nie odnosi się do materiałów źródłowych, nie dostarcza faktów, a jedynie ogólnikowe i entuzjastyczne stwierdzenia, co czyni ją nieakceptowalną jako odpowiedź edukacyjną.
is_accept=False feedback='Odpowiedź modelu jest stylizowana i humorystyczna, ale nie zawiera konkretnych informacji dotyczących Cesarstwa Rzymskiego, nie odnosi się do materiałów źródłowych i nie dostarcza merytorycznych faktów. Ponadto fragment "Caesar nie chodził na spacery z dinozaurami" jest oczywiście anachroniczny i nie ma związku z tematem. W związku z tym odpowiedź nie jest zgodna z dostarczonymi materiałami historycznymi dotyczącymi Cesarstwa Rzymskiego.'
O, obywatelu, choć mój duch rzymski nakazuje mi wielbić chwale Cesarstwa, nie mogę ci dać fałszu! Prawda jest święta jak przysięga legionisty. Powiem więc, że Caesar nie chodził na spacery z dinozaurami ani rzymskie amfiteatry nie były miejscem ta